# DATETIMEEVENTS - Bronze Layer

This notebook ingests the MIMIC-IV `datetimeevents` source table and stores it as a Bronze Delta table.

A small source check is completed before ingestion to confirm the table structure and key identifiers.

In [0]:
from pyspark.sql import functions as F

CATALOG = "workspace"
BRONZE_SCHEMA = "bronze"

SOURCE_PATH = "s3://mimic-iii-demo-bucket/mimic-iv-dataset-demo-2.2-folder/icu/datetimeevents.csv"

datetimeevents = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(SOURCE_PATH)
)

print(f"Rows: {datetimeevents.count():,}")
print(f"Columns: {len(datetimeevents.columns)}")

datetimeevents.printSchema()

### Source Check

`DATETIMEEVENTS` contains ICU events where the recorded value is a date or time.

The main patient, stay, item and caregiver identifiers are checked for missing values before the source data is stored in Bronze.

In [0]:
# Check core fields
key_columns = [
    "subject_id",
    "hadm_id",
    "stay_id",
    "caregiver_id",
    "itemid",
    "charttime"
]

for column in key_columns:
    nulls = datetimeevents.filter(F.col(column).isNull()).count()
    print(f"{column}: {nulls:,} nulls")

### Source Check

The main patient, admission, ICU stay, caregiver and item identifiers are all complete.

The event timestamp is also complete, so the source table is ready to be stored in Bronze without any changes to the original data.

In [0]:
# Add ingestion metadata
datetimeevents_bronze = (
    datetimeevents
    .withColumn("_source_file", F.lit("datetimeevents.csv"))
    .withColumn("_source_module", F.lit("icu"))
    .withColumn("_ingestion_timestamp", F.current_timestamp())
)

# Save as Delta table
(
    datetimeevents_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.datetimeevents_raw")
)

print("Created: workspace.bronze.datetimeevents_raw")

In [0]:
source_count = datetimeevents.count()
bronze_count = spark.table("workspace.bronze.datetimeevents_raw").count()

print(f"Source: {source_count:,}")
print(f"Bronze: {bronze_count:,}")
print(f"Match: {source_count == bronze_count}")

### Bronze Result

`DATETIMEEVENTS` was successfully written to the Bronze layer with all source records preserved.

The core identifiers and event timestamp are complete, and the Bronze row count matches the source.
